## 1. Initialize Project Environment
Import libraries for enrichment analysis and data export.

In [1]:
from __future__ import annotations

import logging
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List

import pandas as pd
import numpy as np

# g:Profiler for automated enrichment analysis
from gprofiler import GProfiler

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")


def locate_repo_root() -> Path:
    """Find the repository root by looking for data folder."""
    here = Path().resolve()
    for base in [here, *here.parents]:
        if (base / "data").exists():
            return base
    raise FileNotFoundError("Could not locate repository root")


REPO_ROOT = locate_repo_root()
ARTIFACTS = REPO_ROOT / "labs/07_network_viz/assignments/artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

logging.info("Repo root: %s", REPO_ROOT)
logging.info("Artifacts directory: %s", ARTIFACTS)

[INFO] Repo root: /home/rbals/git/daha-bdhb/BDHB-lab


[INFO] Artifacts directory: /home/rbals/git/daha-bdhb/BDHB-lab/labs/07_network_viz/assignments/artifacts


## 2. Define Configuration Parameters
Specify which module to analyze and enrichment settings.

In [2]:
@dataclass
class EnrichmentConfig:
    handle: str
    target_module: int = None  # Module to analyze (None = select largest)
    organism: str = "hsapiens"  # Human genes
    sources: List[str] = None  # GO:BP, GO:MF, GO:CC, KEGG, REAC, etc.
    significance_threshold: float = 0.05
    export_dir: Path = None

    def __post_init__(self):
        if self.sources is None:
            self.sources = ["GO:BP", "GO:MF", "GO:CC", "KEGG", "REAC"]
        if self.export_dir is None:
            self.export_dir = ARTIFACTS

    def describe(self) -> Dict[str, str]:
        info = asdict(self)
        info["export_dir"] = str(info["export_dir"])
        return info


CONFIG = EnrichmentConfig(handle="AndreiCod")
CONFIG.describe()

{'handle': 'AndreiCod',
 'target_module': None,
 'organism': 'hsapiens',
 'sources': ['GO:BP', 'GO:MF', 'GO:CC', 'KEGG', 'REAC'],
 'significance_threshold': 0.05,
 'export_dir': '/home/rbals/git/daha-bdhb/BDHB-lab/labs/07_network_viz/assignments/artifacts'}

In [ ]:
# Load module mapping and hub genes from previous tasks
modules_path = ARTIFACTS / f"modules_tp53_{CONFIG.handle}.csv"
hubs_path = ARTIFACTS / f"hubs_tp53_{CONFIG.handle}.csv"

if not modules_path.exists():
    raise FileNotFoundError("Run Task 2 first to generate module mapping.")

modules_df = pd.read_csv(modules_path)
gene2module = dict(zip(modules_df["Gene"], modules_df["Module"]))

if hubs_path.exists():
    hubs_df = pd.read_csv(hubs_path)
else:
    hubs_df = pd.DataFrame(columns=["Gene", "Degree", "Module"])

logging.info(
    "Loaded %d genes across %d modules",
    len(gene2module),
    len(set(gene2module.values())),
)

[INFO] Loaded 80 genes across 4 modules


## 3. Module Analysis
Analyze module composition and select target module for enrichment.

In [ ]:
def get_module_summary(modules_df: pd.DataFrame) -> pd.DataFrame:
    """Summarize module sizes and composition."""
    summary = (
        modules_df.groupby("Module")
        .agg(
            gene_count=("Gene", "count"),
            genes=(
                "Gene",
                lambda x: ", ".join(sorted(x)[:5]) + ("..." if len(x) > 5 else ""),
            ),
        )
        .reset_index()
    )
    return summary.sort_values("gene_count", ascending=False)


def get_module_genes(modules_df: pd.DataFrame, module_id: int) -> List[str]:
    """Get all genes in a specific module."""
    return modules_df[modules_df["Module"] == module_id]["Gene"].tolist()


def get_module_hubs(hubs_df: pd.DataFrame, module_id: int) -> pd.DataFrame:
    """Get hub genes in a specific module."""
    return hubs_df[hubs_df["Module"] == module_id]


# Module overview
module_summary = get_module_summary(modules_df)
print("Module Summary:")
print(module_summary.to_string(index=False))

Module Summary:
 Module  gene_count                                          genes
      0          20  Gene_1, Gene_10, Gene_11, Gene_12, Gene_13...
      1          20 Gene_21, Gene_22, Gene_23, Gene_24, Gene_25...
      2          20 Gene_41, Gene_42, Gene_43, Gene_44, Gene_45...
      3          20 Gene_61, Gene_62, Gene_63, Gene_64, Gene_65...


In [ ]:
# Select target module (largest module by default)
if CONFIG.target_module is None:
    CONFIG.target_module = int(module_summary.iloc[0]["Module"])

target_genes = get_module_genes(modules_df, CONFIG.target_module)
target_hubs = get_module_hubs(hubs_df, CONFIG.target_module)

print(f"\n=== Target Module {CONFIG.target_module} ===")
print(f"Gene count: {len(target_genes)}")
print(f"Hub genes in module: {len(target_hubs)}")
print(
    f"\nGenes: {', '.join(target_genes[:20])}"
    + ("..." if len(target_genes) > 20 else "")
)


=== Target Module 0 ===
Gene count: 20
Hub genes in module: 0

Genes: Gene_1, Gene_10, Gene_11, Gene_12, Gene_13, Gene_14, Gene_15, Gene_16, Gene_17, Gene_18, Gene_19, Gene_2, Gene_20, Gene_3, Gene_4, Gene_5, Gene_6, Gene_7, Gene_8, Gene_9


## 4. Automated Enrichment Analysis with g:Profiler
Use the gprofiler-official Python package to perform GO/KEGG enrichment analysis programmatically.

In [ ]:
def run_gprofiler_enrichment(
    gene_list: List[str],
    organism: str = "hsapiens",
    sources: List[str] = None,
    significance_threshold: float = 0.05,
) -> pd.DataFrame:
    """
    Run g:Profiler functional enrichment analysis.

    Args:
        gene_list: List of gene symbols/IDs
        organism: Organism code (hsapiens, mmusculus, etc.)
        sources: Data sources to query (GO:BP, GO:MF, GO:CC, KEGG, REAC, etc.)
        significance_threshold: P-value cutoff for significance

    Returns:
        DataFrame with enrichment results
    """
    if sources is None:
        sources = ["GO:BP", "GO:MF", "GO:CC", "KEGG", "REAC"]

    gp = GProfiler(return_dataframe=True)

    results = gp.profile(
        organism=organism,
        query=gene_list,
        sources=sources,
        user_threshold=significance_threshold,
        significance_threshold_method="fdr",  # Benjamini-Hochberg correction
        no_evidences=False,
    )

    if results.empty:
        logging.warning("No significant enrichment results found.")
        return results

    # Sort by p-value
    results = results.sort_values("p_value")

    logging.info("Found %d significant enrichment terms", len(results))
    return results

In [ ]:
# Run enrichment analysis on target module
print(f"Running g:Profiler enrichment analysis on Module {CONFIG.target_module}...")
print(f"Gene list: {len(target_genes)} genes")
print(f"Organism: {CONFIG.organism}")
print(f"Sources: {CONFIG.sources}")
print()

enrichment_results = run_gprofiler_enrichment(
    gene_list=target_genes,
    organism=CONFIG.organism,
    sources=CONFIG.sources,
    significance_threshold=CONFIG.significance_threshold,
)

if not enrichment_results.empty:
    print(f"\nFound {len(enrichment_results)} significant enrichment terms!")
else:
    print("\nNo significant enrichment found (this may be due to generic gene names).")

Running g:Profiler enrichment analysis on Module 0...
Gene list: 20 genes
Organism: hsapiens
Sources: ['GO:BP', 'GO:MF', 'GO:CC', 'KEGG', 'REAC']



[WARNING] No significant enrichment results found.



No significant enrichment found (this may be due to generic gene names).


In [ ]:
# Display top enrichment results
if not enrichment_results.empty:
    # Select key columns for display
    display_cols = [
        "source",
        "native",
        "name",
        "p_value",
        "term_size",
        "intersection_size",
    ]
    available_cols = [c for c in display_cols if c in enrichment_results.columns]

    print("\n=== Top 20 Enriched Terms ===")
    print(enrichment_results[available_cols].head(20).to_string(index=False))
else:
    print("No enrichment results to display.")

No enrichment results to display.


In [ ]:
# Summarize by source
if not enrichment_results.empty:
    print("\n=== Enrichment Summary by Source ===")
    source_summary = (
        enrichment_results.groupby("source")
        .agg(
            term_count=("native", "count"),
            min_pvalue=("p_value", "min"),
            top_term=("name", "first"),
        )
        .reset_index()
    )
    print(source_summary.to_string(index=False))

## 5. Diseasome Context
Document how this module might relate to disease networks.

In [10]:
def generate_diseasome_context() -> Dict[str, str]:
    """
    Provide template context for diseasome discussion.

    The diseasome concept (Barabási et al., Nature Genetics 2007) connects
    diseases through shared genes. Co-expression modules can reveal:
    - Functional gene groups relevant to disease
    - Potential drug targets (hub genes)
    - Disease comorbidity patterns
    """
    return {
        "concept": "The human diseasome maps diseases to shared genes, revealing molecular connections between seemingly unrelated conditions.",
        "module_relevance": "Co-expression modules identify functionally related gene groups that may be disrupted in disease states.",
        "hub_significance": "Hub genes in disease-associated modules are often key regulators and potential therapeutic targets.",
        "tp53_context": "TP53 is one of the most connected genes in the diseasome, linked to cancer, aging, and metabolic disorders.",
        "integration": "Modules from TP53-associated networks may reveal pathways involved in tumor suppression, DNA repair, and cell cycle control.",
    }


diseasome_context = generate_diseasome_context()
for key, value in diseasome_context.items():
    print(f"\n{key.upper()}:")
    print(f"  {value}")


CONCEPT:
  The human diseasome maps diseases to shared genes, revealing molecular connections between seemingly unrelated conditions.

MODULE_RELEVANCE:
  Co-expression modules identify functionally related gene groups that may be disrupted in disease states.

HUB_SIGNIFICANCE:
  Hub genes in disease-associated modules are often key regulators and potential therapeutic targets.

TP53_CONTEXT:
  TP53 is one of the most connected genes in the diseasome, linked to cancer, aging, and metabolic disorders.

INTEGRATION:
  Modules from TP53-associated networks may reveal pathways involved in tumor suppression, DNA repair, and cell cycle control.


## 6. Export Results
Save enrichment results and module data for the PDF report.

In [ ]:
# Export target module gene list
genelist_path = CONFIG.export_dir / f"task4_module{CONFIG.target_module}_genes.txt"
with open(genelist_path, "w") as f:
    f.write("\n".join(target_genes))
logging.info("[OK] Gene list saved to: %s", genelist_path.resolve())

# Export module summary
summary_path = CONFIG.export_dir / "task4_module_summary.csv"
module_summary.to_csv(summary_path, index=False)
logging.info("[OK] Module summary saved to: %s", summary_path.resolve())

# Export enrichment results
if not enrichment_results.empty:
    enrichment_path = (
        CONFIG.export_dir / f"task4_enrichment_module{CONFIG.target_module}.csv"
    )
    enrichment_results.to_csv(enrichment_path, index=False)
    logging.info("[OK] Enrichment results saved to: %s", enrichment_path.resolve())

    # Export top terms summary for report
    top_terms_path = CONFIG.export_dir / "task4_top_enriched_terms.csv"
    top_cols = ["source", "native", "name", "p_value", "term_size", "intersection_size"]
    available = [c for c in top_cols if c in enrichment_results.columns]
    enrichment_results[available].head(20).to_csv(top_terms_path, index=False)
    logging.info("[OK] Top enriched terms saved to: %s", top_terms_path.resolve())
else:
    logging.info("No enrichment results to export.")

# Export report metadata
report_data = {
    "module_id": CONFIG.target_module,
    "gene_count": len(target_genes),
    "hub_count": len(target_hubs),
    "enriched_terms": len(enrichment_results) if not enrichment_results.empty else 0,
    "organism": CONFIG.organism,
    "significance_threshold": CONFIG.significance_threshold,
}
report_df = pd.DataFrame([report_data])
report_path = CONFIG.export_dir / "task4_report_data.csv"
report_df.to_csv(report_path, index=False)
logging.info("[OK] Report metadata saved to: %s", report_path.resolve())

print(f"\n✓ Task 4 complete: Enrichment analysis finished.")
if not enrichment_results.empty:
    print(f"  Found {len(enrichment_results)} significant enrichment terms.")
    print(f"  Results exported to artifacts/ folder.")

[INFO] [OK] Gene list saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/07_network_viz/assignments/artifacts/task4_module0_genes.txt


[INFO] [OK] Module summary saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/07_network_viz/assignments/artifacts/task4_module_summary.csv


[INFO] No enrichment results to export.


[INFO] [OK] Report metadata saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/07_network_viz/assignments/artifacts/task4_report_data.csv



✓ Task 4 complete: Enrichment analysis finished.
